In [5]:
import pandas as pd
from bs4 import BeautifulSoup
import json
import re

def is_vietnamese(text):
    """
    Kiểm tra xem chuỗi có chứa kí tự tiếng Việt hay không.
    Bao gồm các chữ có dấu đặc trưng: à á ạ ả ã â ầ ấ ậ ẩ ẫ ă ằ ắ ặ ẳ ẵ...
    """
    vietnamese_pattern = re.compile(
        r'[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ'
        r'ÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴÈÉẸẺẼÊỀẾỆỂỄÌÍỊỈĨÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠÙÚỤỦŨƯỪỨỰỬỮỲÝỴỶỸĐ]',
        re.IGNORECASE
    )
    return bool(vietnamese_pattern.search(text))

def extract_deep_scan(excel_file, output_json):
    print("--- Bắt đầu quét cạn từng dòng (Deep Scan) ---")
    df = pd.read_excel(excel_file)
    final_data = []

    for index, row in df.iterrows():
        name = row['individual']
        html_content = row['html_content']
        soup = BeautifulSoup(html_content, 'html.parser')

        # Loại bỏ các thành phần rác không chứa thông tin hữu ích
        for noise in soup(['script', 'style', 'noscript', 'link']):
            noise.decompose()

        character_entry = {
            "nhan_vat": name,
            "tat_ca_dong_tieng_viet": []
        }

        # Duyệt qua từng node văn bản trong HTML
        # Phương pháp này quét từng hàng chữ xuất hiện trên giao diện
        for text_node in soup.find_all(string=True):
            content = text_node.strip()
            
            # Chỉ lấy nếu là tiếng Việt và có độ dài thực tế
            if len(content) > 1 and is_vietnamese(content):
                # Làm sạch các ký tự đặc biệt thừa
                clean_content = re.sub(r'\[\d+\]', '', content) # Xóa chú thích [1], [2]
                character_entry["tat_ca_dong_tieng_viet"].append(clean_content)

        final_data.append(character_entry)

    # Xuất ra JSON
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)
    
    print(f"--- Hoàn tất! Đã lưu kết quả quét vào file: {output_json} ---")

# Thực thi
extract_deep_scan('individual.xlsx', 'scan_can_nhan_vat.json')

--- Bắt đầu quét cạn từng dòng (Deep Scan) ---
--- Hoàn tất! Đã lưu kết quả quét vào file: scan_can_nhan_vat.json ---
